# Compliance checking of the Boston Zoning Code versions

The Boston Zoning Code is formalized as three MP-Declare models $P_1, P_2, P_3$ (one `.decl` file per version in `mp-declare-models/`).
Following Section 4 of the paper, the event log is split on the enactment dates $t_k$ into sublogs $L_1, L_2, L_3$, where $L_k$ holds the projects whose first event falls in $[t_k, t_{k+1})$.

Each constraint $p$ is checked over its duration: against $L_k$ for every version $V_k$ with $p \in P_k$.
A rule present in $V_1$ and $V_2$ is thus checked on $L_1 \cup L_2$, a rule present only in $V_2$ and $V_3$ on $L_2 \cup L_3$.
We run Declare4Py once per version, with that version's model as written, on its own sublog; a rule's result over its duration is the union of these runs.

A compliance result is `vac` (the constraint never activates), `ful` (activated, all activations fulfilled) or `vio` (at least one activation violated).

In [1]:
import contextlib
import io
import re
import warnings
from datetime import date
from pathlib import Path

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):  # hide the PM4Py banner and import warnings
    import pandas as pd
    from pm4py.objects.log.importer.xes import importer as xes_importer
    from pm4py.objects.log.obj import EventLog
    from Declare4Py.D4PyEventLog import D4PyEventLog
    from Declare4Py.ProcessModels.DeclareModel import DeclareModel
    from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
warnings.filterwarnings("ignore")

LOG_PATH = Path("boston_permitting_clean.xes")
DECL_DIR = Path("mp-declare-models")
VERDICTS = ["vac", "ful", "vio"]

## 1. Load the log

`boston_permitting_clean.xes` is produced from `boston_permitting.xes` by running `1_infer_project_type.py` and `2_clean_event_log.py` in this order.

In [2]:
log = xes_importer.apply(
    str(LOG_PATH),
    variant=xes_importer.Variants.ITERPARSE,
    parameters={"show_progress_bar": False, "timestamp_sort": False},
)
print(len(log), "projects,", sum(len(t) for t in log), "events")

938 projects, 12663 events


## 2. Event attributes used by the conditions

MP-Declare conditions read attributes of the activating event, so the project attributes are copied to every event:

- `gfa`: the gross floor area of the project (`gross_floor_area_sqft`).
- `review_type`: `Large` or `Small` from `project_type`.

In [3]:
for trace in log:
    for event in trace:
        event["gfa"] = trace.attributes["gross_floor_area_sqft"]
        event["review_type"] = trace.attributes["project_type"].split()[0]  # "Large Project" -> "Large"

pd.Series([t.attributes["project_type"] for t in log]).value_counts().to_frame("projects")

,projects
Large Project,494
Small Project,444


## 3. Versions and sublogs

The enactment dates come from the `.decl` file names. A project belongs to the sublog of the version in force at its first event.
Projects left without events by the cleaning have no start date and are not in any sublog.

In [4]:
VERSIONS = pd.DataFrame(
    [
        ("V1", "V1_until_2021-01-13.decl", None),
        ("V2", "V2_2021-01-14_to_2025-08-17.decl", "2021-01-14"),
        ("V3", "V3_from_2025-08-18.decl", "2025-08-18"),
    ],
    columns=["version", "file", "enacted"],
).set_index("version")

bounds = [date.fromisoformat(d) if isinstance(d, str) else date.min for d in VERSIONS["enacted"]] + [date.max]

def start(trace):
    return min(event["time:timestamp"] for event in trace).date()

sublogs = {}
for k, version in enumerate(VERSIONS.index):
    traces = [t for t in log if bounds[k] <= start(t) < bounds[k + 1]]
    sublogs[version] = EventLog(
        traces,
        attributes=log.attributes,
        extensions=log.extensions,
        omni_present=log.omni_present,
        classifiers=log.classifiers,
        properties=log.properties,  # Declare4Py reads the timestamp key from here
    )

VERSIONS["projects"] = [len(sublogs[v]) for v in VERSIONS.index]
VERSIONS

,file,enacted,projects
version,,,
V1,V1_until_2021-01-13.decl,NaN,586
V2,V2_2021-01-14_to_2025-08-17.decl,2021-01-14,291
V3,V3_from_2025-08-18.decl,2025-08-18,61


## 4. Rules and their duration

Two constraints are the same rule if their text is identical; a modified rule (such as the PDA decision window, whose time condition changes in $V_2$) is a new constraint.
Rules are numbered in the order they first appear, so the rules of the $V_1$ file come first and the rules added in later versions continue the numbering.
The group is the comment heading under which the rule first appears.

In [5]:
CONSTRAINT = re.compile(r"^[A-Z][A-Za-z0-9 ]*\[")

def read_rules(text):
    """(constraint, group) for each constraint line, in file order."""
    group, rules = None, []
    for line in map(str.strip, text.splitlines()):
        if line.startswith("# ---"):
            group = re.sub(r"^#\s*-+\s*(R\d+-R\d+\s*)?", "", line)
        elif CONSTRAINT.match(line):
            rules.append((line, group))
    return rules

decl_text = {v: (DECL_DIR / f).read_text() for v, f in VERSIONS["file"].items()}
occurrences = pd.DataFrame(
    [(c, g, v) for v, text in decl_text.items() for c, g in read_rules(text)],
    columns=["constraint", "group", "version"],
)
rules = occurrences.drop_duplicates("constraint")[["constraint", "group"]].reset_index(drop=True)
rules.index = [f"R{i + 1:02d}" for i in range(len(rules))]
rule_id = pd.Series(rules.index, index=rules["constraint"])

def span(versions):
    """'V1-V3' for consecutive versions, otherwise a list."""
    ks = sorted(int(v[1:]) for v in versions)
    if ks == list(range(ks[0], ks[-1] + 1)) and len(ks) > 1:
        return f"V{ks[0]}–V{ks[-1]}"
    return ", ".join(f"V{k}" for k in ks)

presence = occurrences.assign(rule=occurrences["constraint"].map(rule_id)).groupby("rule")["version"].agg(list)
rules["versions"] = presence.map(span)
rules["projects"] = presence.map(lambda vs: sum(len(sublogs[v]) for v in vs))

with pd.option_context("display.max_colwidth", None):
    display(rules)

,constraint,group,versions,projects
R01,"Not Responded Existence[Project Notification Form PNF, Small Project Review Application SPRA] | | |","review-type selection (80B-2, 80E-2)",V1–V3,938
R02,Absence1[Small Project Review Application SPRA] |A.gfa >= 50000 |,"review-type selection (80B-2, 80E-2)",V1–V3,938
R03,Absence1[Project Notification Form PNF] |A.gfa < 10000 |,"review-type selection (80B-2, 80E-2)",V1–V3,938
R04,"Response[PNF Comment Period Open, PNF Comment Period End] | | |30,36500,d","Large Project Review (80B-5, 80A-2)",V1–V3,938
R05,"Precedence[Project Notification Form PNF, BPDA Board Approval] |A.review_type is Large | |","Large Project Review (80B-5, 80A-2)",V1–V3,938
R06,"Precedence[BPDA Board Approval, Scoping Determination Waiving Further Review] | | |","Large Project Review (80B-5, 80A-2)",V1–V3,938
R07,"Not Response[Scoping Determination Waiving Further Review, Draft Project Impact Report DPIR] | | |","Large Project Review (80B-5, 80A-2)",V1–V2,877
R08,"Precedence[Scoping Determination, Draft Project Impact Report DPIR] | | |","Large Project Review (80B-5, 80A-2)",V1–V2,877
R09,"Response[DPIR Comment Period Open, DPIR Comment Period End] | | |45,36500,d","Large Project Review (80B-5, 80A-2)",V1–V2,877
R10,"Precedence[Draft Project Impact Report DPIR, Preliminary Adequacy Determination PAD] | | |","Large Project Review (80B-5, 80A-2)",V1–V2,877


## 5. Compliance check

Each version's model is checked on its own sublog. `num_activations == 0` gives `vac`; otherwise the state gives `ful` or `vio`.

In [6]:
frames = []
for version, text in decl_text.items():
    constraints = [c for c, _ in read_rules(text)]
    model = DeclareModel().parse_from_string(text)
    assert [str(c) for c in model.serialized_constraints] == constraints, version
    results = MPDeclareAnalyzer(
        log=D4PyEventLog(log=sublogs[version]), declare_model=model, consider_vacuity=True
    ).run()
    case_ids = [t.attributes["concept:name"] for t in sublogs[version]]
    ids = [rule_id[c] for c in constraints]
    state = results.get_metric("state").set_axis(case_ids).set_axis(ids, axis=1)
    activations = results.get_metric("num_activations").set_axis(case_ids).set_axis(ids, axis=1)
    verdict = state.replace({1: "ful", 0: "vio"}).mask(activations == 0, "vac")
    frames.append(
        verdict.stack().rename("verdict").rename_axis(["case", "rule"]).reset_index().assign(version=version)
    )

verdicts = pd.concat(frames, ignore_index=True)
verdicts.groupby("version")["case"].nunique().to_frame("projects checked")

,projects checked
version,
V1,586
V2,291
V3,61


## 6. Results per rule over its duration

Counts over $L_j \cup \dots \cup L_k$ for the versions in which the rule is present. **Compliant %** is the share of `ful` among the activated projects (`ful` + `vio`).

In [7]:
pooled = verdicts.groupby(["rule", "verdict"]).size().unstack(fill_value=0).reindex(columns=VERDICTS, fill_value=0)
activated = pooled["ful"] + pooled["vio"]
summary = rules[["constraint", "versions", "projects"]].join(pooled)
summary["compliant %"] = (pooled["ful"] / activated).where(activated > 0)
assert (summary[VERDICTS].sum(axis=1) == summary["projects"]).all()

with pd.option_context("display.max_colwidth", None):
    display(summary.style.format({"compliant %": "{:.0%}"}, na_rep="—"))

,constraint,versions,projects,vac,ful,vio,compliant %
R01,"Not Responded Existence[Project Notification Form PNF, Small Project Review Application SPRA] | | |",V1–V3,938,531,404,3,99%
R02,Absence1[Small Project Review Application SPRA] |A.gfa >= 50000 |,V1–V3,938,0,917,21,98%
R03,Absence1[Project Notification Form PNF] |A.gfa < 10000 |,V1–V3,938,0,938,0,100%
R04,"Response[PNF Comment Period Open, PNF Comment Period End] | | |30,36500,d",V1–V3,938,610,321,7,98%
R05,"Precedence[Project Notification Form PNF, BPDA Board Approval] |A.review_type is Large | |",V1–V3,938,546,327,65,83%
R06,"Precedence[BPDA Board Approval, Scoping Determination Waiving Further Review] | | |",V1–V3,938,749,161,28,85%
R07,"Not Response[Scoping Determination Waiving Further Review, Draft Project Impact Report DPIR] | | |",V1–V2,877,694,183,0,100%
R08,"Precedence[Scoping Determination, Draft Project Impact Report DPIR] | | |",V1–V2,877,791,66,20,77%
R09,"Response[DPIR Comment Period Open, DPIR Comment Period End] | | |45,36500,d",V1–V2,877,811,63,3,95%
R10,"Precedence[Draft Project Impact Report DPIR, Preliminary Adequacy Determination PAD] | | |",V1–V2,877,869,7,1,88%


## 7. Results per rule and version

The same counts split by sublog; `—` marks versions in which the rule is not in force.

In [8]:
per_version = (
    verdicts.groupby(["rule", "version", "verdict"]).size()
    .unstack(["version", "verdict"], fill_value=0)
    .reindex(columns=pd.MultiIndex.from_product([VERSIONS.index, VERDICTS]), fill_value=0)
    .astype(float)
)
for version in VERSIONS.index:  # — where the rule is not in force
    per_version.loc[~presence.map(lambda vs: version in vs), version] = float("nan")
per_version.style.format(precision=0, na_rep="—")

Compliant % per version, to compare how practice develops across the versions in which a rule stays the same.

In [9]:
ful = per_version.xs("ful", axis=1, level=1)
vio = per_version.xs("vio", axis=1, level=1)
rate = (ful / (ful + vio)).where(ful + vio > 0)
rate.style.format("{:.0%}", na_rep="—")

version,V1,V2,V3
rule,,,
R01,99%,100%,100%
R02,98%,97%,97%
R03,100%,100%,100%
R04,96%,100%,100%
R05,82%,89%,100%
R06,90%,74%,83%
R07,100%,100%,—
R08,72%,100%,—
R09,96%,93%,—


## Notes

- **Unary templates.** For `Absence1[...]` (the event type occurs fewer than once), Declare4Py never reports `vac`: a project without a matching event fulfills the constraint. The `vac` column of these rules is therefore always 0, unlike the binary templates, where `vac` means that the activation never occurred.
- **BCDC review.** `2_clean_event_log.py` merges the Design Committee presentations, Monthly Meeting presentations, and BCDC approvals into the single event type `Boston Civic Design Commission`, so the BCDC rules count any of them.
- **Snapshot.** Ongoing projects may still change their compliance results, in particular in $L_3$.